# D3: Alerts & Monitoring

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Build monitoring systems** for permit pipeline changes
2. **Create alerts** for stalled projects and milestones
3. **Track status changes** over time
4. **Automate notifications** for stakeholders

## Why This Matters

Housing advocates, journalists, and policymakers need to stay informed:
- When does a major project get approved?
- Which projects have stalled?
- Are there unusual patterns in approvals/denials?

Automated monitoring turns passive data into active intelligence.

## Alert Types

| Alert | Trigger | Audience |
|-------|---------|----------|
| Major approval | Project >50 units approved | City Council, media |
| Stalled project | No change >180 days | Planning staff |
| RHNA milestone | 10% progress increment | Housing advocates |
| Denial spike | >3 denials in month | Policy analysts |

---

## Overview

Track status changes and generate alerts for monitoring.

**Alert Types:**
- Status changes
- Stalled project warnings
- Inspection delay alerts
- Milestone notifications

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Find project root and setup environment
def find_project_root():
    """Find project root by looking for marker directories."""
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() and (path / 'modules').exists():
            return path
    raise FileNotFoundError("Could not find project root")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

# Load config with resolved paths
import json
with open(ROOT / '00_config/berkeley_config.json') as f:
    CONFIG = json.load(f)

# Resolve relative paths to absolute
for key, value in CONFIG['paths'].items():
    if isinstance(value, str) and not value.startswith('http'):
        CONFIG['paths'][key] = str(ROOT / value)

print(f"✅ Project root: {ROOT}")
print(f"✅ Housing data: {CONFIG['paths']['housing_projects']}")

## 2. Load Data

In [ ]:
# Load housing projects
housing_path = Path(CONFIG['paths']['housing_projects'])
df = load_csv(housing_path)

if df is not None:
    print(f"Loaded {len(df)} projects")
    print(f"Total units: {df['net_units'].sum():,.0f}")

## 3. Alert: Potentially Stalled Projects

In [ ]:
# Identify potentially stalled projects
if df is not None:
    # Projects in review status for extended time (based on year)
    current_year = 2025
    df['years_since_filing'] = current_year - df['year']
    
    review_statuses = ['In Review', 'Under Review', 'Incomplete Pending Applicant', 'Corrections Pending Applicant']
    in_review = df[df['status'].isin(review_statuses)].copy()
    
    # Flag as potentially stalled if in review for 2+ years
    potentially_stalled = in_review[in_review['years_since_filing'] >= 2]
    
    print("ALERT: Potentially Stalled Projects")
    print("="*60)
    print(f"Projects in review for 2+ years: {len(potentially_stalled)}")
    print(f"Total units affected: {potentially_stalled['net_units'].sum():,.0f}")
    
    if len(potentially_stalled) > 0:
        print("\nLargest potentially stalled:")
        display(potentially_stalled.nlargest(10, 'net_units')[['address_display', 'net_units', 'status', 'year']])

## 4. Alert: Projects Near Completion

In [ ]:
# Projects nearing completion
if df is not None:
    completion_statuses = ['Pending Final Action', 'Approved']
    near_completion = df[df['status'].str.contains('|'.join(completion_statuses), case=False, na=False)]
    
    print("ALERT: Projects Approaching Completion")
    print("="*60)
    print(f"Projects in final stages: {len(near_completion)}")
    print(f"Total units: {near_completion['net_units'].sum():,.0f}")
    
    if len(near_completion) > 0:
        print("\nLargest projects near completion:")
        display(near_completion.nlargest(10, 'net_units')[['address_display', 'net_units', 'status']])

## 5. Alert: Large Projects to Watch

In [ ]:
# Monitor large projects (100+ units)
if df is not None:
    large_projects = df[df['net_units'] >= 100].copy()
    
    print("MONITORING: Large Projects (100+ units)")
    print("="*60)
    print(f"Total large projects: {len(large_projects)}")
    print(f"Total units: {large_projects['net_units'].sum():,.0f}")
    
    # By status
    print("\nBy Status:")
    status_summary = large_projects.groupby('status').agg({
        'address_display': 'count',
        'net_units': 'sum'
    })
    status_summary.columns = ['Projects', 'Units']
    display(status_summary.sort_values('Units', ascending=False))

## 6. Alert: Appeals and Holds

In [ ]:
# Projects with appeals or holds
if df is not None:
    appeal_keywords = ['Appeal', 'Hold', 'Denied']
    appealed = df[df['status'].str.contains('|'.join(appeal_keywords), case=False, na=False)]
    
    print("ALERT: Projects with Appeals/Holds")
    print("="*60)
    print(f"Projects with appeals: {len(appealed)}")
    print(f"Total units affected: {appealed['net_units'].sum():,.0f}")
    
    if len(appealed) > 0:
        display(appealed[['address_display', 'net_units', 'status']])

## 7. Generate Alerts Summary

In [ ]:
# Create alerts summary
if df is not None:
    alerts = {
        'generated_at': datetime.now().isoformat(),
        'stalled': {
            'count': len(potentially_stalled) if 'potentially_stalled' in dir() else 0,
            'units': int(potentially_stalled['net_units'].sum()) if 'potentially_stalled' in dir() else 0
        },
        'near_completion': {
            'count': len(near_completion) if 'near_completion' in dir() else 0,
            'units': int(near_completion['net_units'].sum()) if 'near_completion' in dir() else 0
        },
        'large_projects': {
            'count': len(large_projects) if 'large_projects' in dir() else 0,
            'units': int(large_projects['net_units'].sum()) if 'large_projects' in dir() else 0
        },
        'appealed': {
            'count': len(appealed) if 'appealed' in dir() else 0,
            'units': int(appealed['net_units'].sum()) if 'appealed' in dir() else 0
        }
    }
    
    # Save alerts
    alerts_path = DATA_DIR / 'alerts_summary.json'
    with open(alerts_path, 'w') as f:
        json.dump(alerts, f, indent=2)
    
    print(f"Alerts saved: {alerts_path}")
    print("\nAlerts Summary:")
    print(json.dumps(alerts, indent=2))

## 8. Export Watch List

In [ ]:
# Create watch list CSV
if df is not None:
    watch_list = []
    
    # Add stalled
    if 'potentially_stalled' in dir() and len(potentially_stalled) > 0:
        for _, row in potentially_stalled.iterrows():
            watch_list.append({
                'address': row['address_display'],
                'units': row['net_units'],
                'status': row['status'],
                'alert_type': 'Potentially Stalled',
                'priority': 'High' if row['net_units'] >= 100 else 'Medium'
            })
    
    # Add appealed
    if 'appealed' in dir() and len(appealed) > 0:
        for _, row in appealed.iterrows():
            watch_list.append({
                'address': row['address_display'],
                'units': row['net_units'],
                'status': row['status'],
                'alert_type': 'Appeal/Hold',
                'priority': 'High'
            })
    
    if watch_list:
        watch_df = pd.DataFrame(watch_list)
        watch_path = DATA_DIR / 'project_watch_list.csv'
        watch_df.to_csv(watch_path, index=False)
        print(f"Watch list saved: {watch_path}")
        print(f"Total items: {len(watch_list)}")
    else:
        print("No items for watch list")

---

## Summary

This notebook:
- Identified potentially stalled projects
- Tracked projects near completion
- Monitored large projects
- Flagged appeals and holds
- Generated alerts summary and watch list

**Workflow Complete!**

The Berkeley Housing Data Pipeline is now fully organized into:
- A: Data Collection (A1-A3)
- B: Timeline Tracking (B1-B3)
- C: Analysis (C1-C3)
- D: Reporting (D1-D3)